In [4]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

import gc
import time


import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
from statistics import mean, median, stdev

from complex_networks.io import load_edgelist, preprocess
#from src.complex_networks.config import DATA_DIR, RAW_DATA_DIR, GRAPHML_DATA_DIR
from complex_networks.config import *


# CSV to TXT

In [ ]:
nome = "HU_edges.csv"

df = pd.read_csv(RAW_DATA_DIR / nome)

print(df)

G = nx.from_pandas_edgelist(
    df,
    source="node_1",
    target="node_2",
    create_using=nx.Graph,
)

print(f"{G.number_of_nodes()} nós, {G.number_of_edges()} arestas")

nx.write_edgelist(
    G,
    RAW_DATA_DIR / "gemsec-deezer-hu.txt.gz",
    data=False,
)

del G, df
gc.collect()

In [ ]:
for arquivo in sorted(RAW_DATA_DIR.glob("*.csv")):
    if not arquivo.is_file():
        continue

    codigo = arquivo.stem.lower().removesuffix("_edges")
    destino = RAW_DATA_DIR / f"gemsec-Facebook-{codigo}.txt.gz"

    if destino.exists():
        print(f"{destino.name} já existe")
        continue

    df = pd.read_csv(arquivo)

    G = nx.from_pandas_edgelist(
        df,
        source="node_1",
        target="node_2",
        create_using=nx.Graph,
    )

    print(
        f"{arquivo.name}: "
        f"{G.number_of_nodes()} nós, "
        f"{G.number_of_edges()} arestas"
    )

    nx.write_edgelist(G, destino, data=False)

    print(f"Salvo: {destino.name}")

    del G, df
    gc.collect()

# CSV to GRAPHML

In [ ]:
nome = "soc-pokec-relationships"

G = nx.read_edgelist(RAW_DATA_DIR / f"{nome}.txt.gz", data=False) 

print(f"{nome} em memoria")

nx.write_graphml(G,GRAPHML_DATA_DIR / f"{nome}.graphml")

print(f"{nome} gravado")

del G
gc.collect()
time.sleep(60)

In [ ]:
for f in RAW_DATA_DIR.iterdir():
    if f.is_file():
        G = nx.read_edgelist(f, data=False)

        name = f.name.split(".")[0]
        print(name)

        nx.write_graphml(G, GRAPHML_DATA_DIR / f"{name}.graphml")
        
        del G
        gc.collect()
        time.sleep(60)

In [ ]:
arquivos = sorted(
    (p for p in RAW_DATA_DIR.glob("*.txt.gz") if p.is_file()),
    key=lambda p: p.stat().st_size,
)

arquivos

In [ ]:
# /home/fernando/complex-networks/data/raw/com-orkut.txt.gz
# /home/fernando/complex-networks/data/raw/soc-LiveJournal1.txt.gz
# /home/fernando/complex-networks/data/raw/soc-pokec-relationships.txt.gz
# /home/fernando/complex-networks/data/raw/wiki-topcats.txt.gz

In [ ]:
arquivos = sorted(
    (p for p in RAW_DATA_DIR.glob("*.txt.gz") if p.is_file()),
    key=lambda p: p.stat().st_size,
)

already_converted = {
    p.stem
    for p in GRAPHML_DATA_DIR.glob("*.graphml")
    if p.is_file()
}

for arquivo in arquivos:
    nome = arquivo.name.removesuffix(".txt.gz")

    if nome in already_converted:
        print(f"|{nome}| ja convertido")
        continue

    G = nx.read_edgelist(arquivo, data=False)

    print(f"{nome} lido na memória")

    nx.write_graphml(G, GRAPHML_DATA_DIR / f"{nome}.graphml")

    print(f"{nome} salvo em disco")

    del G
    gc.collect()

In [ ]:
for arquivo in sorted(RAW_DATA_DIR.glob("*.txt.gz")):
    G = load_edgelist(arquivo)

    print(f"{arquivo.name}: {nx.number_of_selfloops(G)} self-loops")

    del G
    gc.collect()

In [ ]:
from src.complex_networks.io import (
    load_edgelist,
    preprocess
)
from src.complex_networks.metrics import (
    calculate_general_metrics,
    get_degree_distribution,
    get_clustering_distribution,
    get_component_size_distribution,
    estimate_average_distance
)
from src.complex_networks.plots import (
    plot_degree_distribution,
    plot_clustering_distribution,
    plot_component_size_distribution,
)
from src.complex_networks.config import RAW_DATA_DIR, RESULTS_DIR

In [ ]:


name = "p2p-Gnutella04"
print("-----------------")
print(f"| {name} |")
print("-----------------")
print("=====Carregando grafo=====")
G = load_edgelist(RAW_DATA_DIR / f"{name}.txt.gz")
G = preprocess(G)


print("=====Calculando métricas gerais=====")
row = calculate_general_metrics(G, name, RESULTS_DIR / "metrics")


print("=====Distribuição de grau=====")
plot_degree_distribution(
    get_degree_distribution(G),
    RESULTS_DIR / "distributions/degree" / f"{name}.png",
    title=name,
    log_scale=True,
)

print("=====Distribuição de coeficiente de clustering=====")
plot_clustering_distribution(
    get_clustering_distribution(G),
    RESULTS_DIR / "distributions/clustering" / f"{name}.png",
    title=name,
)


print("=====Distribuição de componentes=====")
if row["n_components"] > 1:
    plot_component_size_distribution(
        get_component_size_distribution(G),
        RESULTS_DIR / "distributions/components" / f"{name}.png",
        title=name,
        log_scale=True,
    )

del row, G
gc.collect()

In [ ]:

files = sorted(
    RAW_DATA_DIR.glob("*.txt.gz"),
    key=lambda path: path.stat().st_size,
)

for path in files[35:]:
    name = path.name.removesuffix(".txt.gz")
    size_mb = path.stat().st_size / (1024 ** 2)

    print(f"\n===== {name} | {size_mb:.2f} MB =====")

    print("=====Carregando grafo=====")
    G = load_edgelist(path)
    G = preprocess(G)

    print("=====Calculando métricas gerais=====")
    row = calculate_general_metrics(
        G, name, RESULTS_DIR / "metrics", estimate_distance=True, estimate_diameter=True
    )

    print("=====Distribuição de grau=====")
    plot_degree_distribution(
        get_degree_distribution(G),
        RESULTS_DIR / "distributions/degree" / f"re-{name}.png",
        title=name,
        log_scale=True,
    )

    print("=====Distribuição de coeficiente de clustering=====")
    plot_clustering_distribution(
        get_clustering_distribution(G),
        RESULTS_DIR / "distributions/clustering" / f"re-{name}.png",
        title=name,
    )

    if row["n_components"] > 1:
        print("=====Distribuição de componentes=====")
        plot_component_size_distribution(
            get_component_size_distribution(G),
            RESULTS_DIR / "distributions/components" / f"re-{name}.png",
            title=name,
            log_scale=True,
        )

    del row, G
    gc.collect()

In [ ]:
files = sorted(
    RAW_DATA_DIR.glob("*.txt.gz"),
    key=lambda path: path.stat().st_size,
)

files[34:]

In [ ]:
G = load_edgelist(RAW_DATA_DIR / "email-EuAll.txt.gz")
G = preprocess(G)
largest_component = max(nx.connected_components(G), key=len)
G = G.subgraph(largest_component).copy()

estimates = []

print("Comecando a estimativa")
for seed in range(1, 6):
    value = estimate_average_distance(G, percentage=1, seed=seed)
    estimates.append(value)

    print(f"Seed {seed:2d}: {value:.6f}")

print("\nResumo das 10 estimativas:")
print(f"Média:         {mean(estimates):.6f}")
print(f"Mediana:       {median(estimates):.6f}")
print(f"Desvio padrão: {stdev(estimates):.6f}")

In [4]:
files = sorted(
    RAW_DATA_DIR.glob("*.txt.gz"),
    key=lambda path: path.stat().st_size,
)

for path in files:
    G = load_edgelist(path)

    total = G.number_of_nodes()
    degree_zero = sum(1 for _, d in G.degree() if d == 0)
    degree_pos = total - degree_zero

    pct_zero_vs_pos = (degree_zero / degree_pos * 100) if degree_pos else float("inf")
    pct_zero_total = (degree_zero / total * 100) if total else 0.0

    print(
        f"{path.name}: "
        f"total={total}, grau=0: {degree_zero}, grau>0: {degree_pos} | "
        f"zero/(grau>0) = {pct_zero_vs_pos:.2f}% | "
        f"zero/total = {pct_zero_total:.2f}%"
    )

    del G
    gc.collect()

gemsec-Facebook-tvshow.txt.gz: total=3892, grau=0: 0, grau>0: 3892 | zero/(grau>0) = 0.00% | zero/total = 0.00%
p2p-Gnutella04.txt.gz: total=10876, grau=0: 0, grau>0: 10876 | zero/(grau>0) = 0.00% | zero/total = 0.00%
gemsec-Facebook-politician.txt.gz: total=5908, grau=0: 0, grau>0: 5908 | zero/(grau>0) = 0.00% | zero/total = 0.00%
p2p-Gnutella25.txt.gz: total=22687, grau=0: 0, grau>0: 22687 | zero/(grau>0) = 0.00% | zero/total = 0.00%
gemsec-Facebook-company.txt.gz: total=14113, grau=0: 0, grau>0: 14113 | zero/(grau>0) = 0.00% | zero/total = 0.00%
p2p-Gnutella24.txt.gz: total=26518, grau=0: 0, grau>0: 26518 | zero/(grau>0) = 0.00% | zero/total = 0.00%
gemsec-Facebook-public_figure.txt.gz: total=11565, grau=0: 0, grau>0: 11565 | zero/(grau>0) = 0.00% | zero/total = 0.00%
gemsec-Facebook-government.txt.gz: total=7057, grau=0: 0, grau>0: 7057 | zero/(grau>0) = 0.00% | zero/total = 0.00%
p2p-Gnutella30.txt.gz: total=36682, grau=0: 0, grau>0: 36682 | zero/(grau>0) = 0.00% | zero/total = 0.

In [5]:
files = sorted(
    RAW_DATA_DIR.glob("*.txt.gz"),
    key=lambda path: path.stat().st_size,
)

for path in files:
    G = load_edgelist(path)

    n_loops = nx.number_of_selfloops(G)
    total = G.number_of_edges()
    pct_total = (n_loops / total * 100)

    print(
        f"{path.name}: "
        f"total={total}, self-loops: {n_loops} | "
        f"self_loops/total = {pct_total:.2f}%"
    )

    del G
    gc.collect()

gemsec-Facebook-tvshow.txt.gz: total=17262, self-loops: 23 | self_loops/total = 0.13%
p2p-Gnutella04.txt.gz: total=39994, self-loops: 0 | self_loops/total = 0.00%
gemsec-Facebook-politician.txt.gz: total=41729, self-loops: 23 | self_loops/total = 0.06%
p2p-Gnutella25.txt.gz: total=54705, self-loops: 0 | self_loops/total = 0.00%
gemsec-Facebook-company.txt.gz: total=52310, self-loops: 184 | self_loops/total = 0.35%
p2p-Gnutella24.txt.gz: total=65369, self-loops: 0 | self_loops/total = 0.00%
gemsec-Facebook-public_figure.txt.gz: total=67114, self-loops: 76 | self_loops/total = 0.11%
gemsec-Facebook-government.txt.gz: total=89455, self-loops: 26 | self_loops/total = 0.03%
p2p-Gnutella30.txt.gz: total=88328, self-loops: 0 | self_loops/total = 0.00%
gemsec-Facebook-athletes.txt.gz: total=86858, self-loops: 47 | self_loops/total = 0.05%
p2p-Gnutella31.txt.gz: total=147892, self-loops: 0 | self_loops/total = 0.00%
gemsec-Deezer-ro.txt.gz: total=125826, self-loops: 0 | self_loops/total = 0.00%

In [6]:
from src.complex_networks.fitting import fit_degree_distributions

files = sorted(
    RAW_DATA_DIR.glob("*.txt.gz"),
    key=lambda path: path.stat().st_size,
)

file = files[0]

G = load_edgelist(file)

result_full = fit_degree_distributions(G, xmin=1)

# Cauda com início estimado automaticamente pela lei de potência:
result_tail = fit_degree_distributions(G, xmin=None)


del G


print(f'Results tail {result_tail["xmin"]}')
print(f'Results tail {result_tail["fitted_percentage"]}')

# Comparação dos cinco modelos:
df_full = pd.DataFrame(result_full["models"])

df_tail = pd.DataFrame(result_tail["models"])



dist: <powerlaw.distributions.Power_Law object at 0x7f9243d27890>
dist: <powerlaw.distributions.Exponential object at 0x7f9243d27750>
dist: <powerlaw.distributions.Lognormal object at 0x7f9243d27610>
dist: <powerlaw.distributions.Stretched_Exponential object at 0x7f9243d274d0>
dist: <powerlaw.distributions.Truncated_Power_Law object at 0x7f9243d27250>
dist: <powerlaw.distributions.Power_Law object at 0x7f9243d26fd0>
dist: <powerlaw.distributions.Exponential object at 0x7f9243d265d0>
dist: <powerlaw.distributions.Lognormal object at 0x7f9243d26710>
dist: <powerlaw.distributions.Stretched_Exponential object at 0x7f9243d26850>
dist: <powerlaw.distributions.Truncated_Power_Law object at 0x7f9243d26ad0>
Results tail 17
Results tail 13.617677286742035


In [4]:
df_full

,model,parameters,log_likelihood,aic,ks,delta_aic
0,lognormal,"{'mu': 1.4830850468836134, 'sigma': 1.17269505...",-11865.167713,23734.335427,0.008496,0.000000
1,stretched_exponential,"{'Lambda': 0.17366317197891956, 'beta': 0.6954...",-11905.289255,23814.578510,0.046210,80.243083
2,truncated_power_law,"{'alpha': 0.5867640817646949, 'Lambda': 0.0642...",-11941.539232,23887.078464,0.059581,152.743038
3,exponential,{'Lambda': 0.1196028487429035},-12159.070099,24320.140198,0.110260,585.804771
4,power_law,{'alpha': 1.4813498342201101},-12958.480523,25918.961046,0.215980,2184.625619


In [5]:
df_tail

,model,parameters,log_likelihood,aic,ks,delta_aic
0,truncated_power_law,"{'alpha': 1.5914460748673434, 'Lambda': 0.0234...",-2009.563752,4023.127503,0.054443,0.000000
1,stretched_exponential,"{'Lambda': 0.36933311108732325, 'beta': 0.5195...",-2011.118077,4026.236155,0.056398,3.108652
2,lognormal,"{'mu': 2.0922595181494152, 'sigma': 0.97975153...",-2012.758957,4029.517914,0.056271,6.390411
3,exponential,{'Lambda': 0.05975626632224358},-2023.164518,4048.329037,0.097236,25.201534
4,power_law,{'alpha': 2.747546598156399},-2023.820427,4049.640854,0.044017,26.513351


In [ ]:
# KS pior afastamento acumulado encontrado

In [2]:
from src.complex_networks.io import get_network_path, load_edgelist

G = load_edgelist(get_network_path("com-dblp"))


In [3]:
from src.complex_networks.fitting import fit_degree_distributions

result_full = fit_degree_distributions(G, xmin=1)

# Cauda com início estimado automaticamente pela lei de potência:
result_tail = fit_degree_distributions(G, xmin=None)

result_after_peak = fit_degree_distributions(G, xmin=5)

print(f'Results tail {result_tail["xmin"]}')
print(f'Results tail {result_tail["fitted_percentage"]}')

# Comparação dos cinco modelos:
df_full = pd.DataFrame(result_full["models"])

df_tail = pd.DataFrame(result_tail["models"])

df_peak = pd.DataFrame(result_after_peak["models"])

/home/fernando/complex-networks/.venv/lib/python3.14/site-packages/powerlaw/distributions.py:2053: RuntimeWarning: divide by zero encountered in scalar divide
  C = 1.0/C
/home/fernando/complex-networks/.venv/lib/python3.14/site-packages/scipy/optimize/_optimize.py:870: RuntimeWarning: invalid value encountered in subtract
  np.max(np.abs(fsim[0] - fsim[1:])) <= fatol):


Results tail 29
Results tail 3.168285606156175


In [4]:
result_full

{'powerlaw_version': '2.0.0',
 'xmin_requested': 1,
 'xmin': 1,
 'n_nodes': 317080,
 'n_fitted': 317080,
 'fitted_percentage': 100.0,
 'discrete': True,
 'discrete_normalization': 'round',
 'fit_method': 'likelihood',
 'distance_metrics_source': 'powerlaw.compute_distance_metrics',
 'models': [{'model': 'power_law',
   'parameters': {'alpha': 1.532202627477247},
   'parameter_ranges': {'alpha': (1.000001, None)},
   'n_parameters': 1,
   'log_likelihood': -964468.173368971,
   'aic': 1928938.346737942,
   'ks': 0.2648536926415743,
   'kuiper': 0.3612442684859485,
   'anderson_darling': 12.727790551510552,
   'kappa': 0.9406956737753938,
   'in_range': True,
   'noise_flag': False},
  {'model': 'exponential',
   'parameters': {'Lambda': 0.16370695425356666},
   'parameter_ranges': {'Lambda': [0, None]},
   'n_parameters': 1,
   'log_likelihood': -891245.1194182658,
   'aic': 1782492.2388365315,
   'ks': 0.11084534255458622,
   'kuiper': 0.13273844405087665,
   'anderson_darling': 291942

In [5]:
df_full

,model,parameters,parameter_ranges,n_parameters,log_likelihood,aic,ks,kuiper,anderson_darling,kappa,in_range,noise_flag
0,power_law,{'alpha': 1.532202627477247},"{'alpha': (1.000001, None)}",1,-964468.173369,1.928938e+06,0.264854,0.361244,12.727791,0.940696,True,False
1,exponential,{'Lambda': 0.16370695425356666},"{'Lambda': [0, None]}",1,-891245.119418,1.782492e+06,0.110845,0.132738,291942.888983,0.999982,True,False
2,lognormal,"{'mu': 1.3300826132576185, 'sigma': 0.99321572...","{'mu': [None, None], 'sigma': [0, None]}",2,-863364.576974,1.726733e+06,0.051134,0.073778,0.350993,1.001839,True,False
3,stretched_exponential,"{'Lambda': 0.22096197538381346, 'beta': 0.7481...","{'Lambda': [0, None], 'beta': [0, None]}",2,-877615.584339,1.755235e+06,0.073262,0.134278,8.667928,0.998955,True,False
4,truncated_power_law,"{'alpha': 0.4776801110110386, 'Lambda': 0.1036...","{'alpha': (0.0, None), 'Lambda': [0, None]}",2,-882538.581627,1.765081e+06,0.087509,0.145142,1282.228440,0.998536,True,False


In [6]:
df_tail

,model,parameters,parameter_ranges,n_parameters,log_likelihood,aic,ks,ks_plus,ks_minus,kuiper,anderson_darling,kappa,in_range,noise_flag
0,power_law,{'alpha': 3.2625800229227178},"{'alpha': (1.000001, None)}",1,-39941.946556,79885.893111,0.034284,0.034284,0.015188,0.049472,0.811096,0.995726,True,False
1,exponential,{'Lambda': 0.05022453944284436},"{'Lambda': [0, None]}",1,-40097.480703,80196.961406,0.066249,0.026768,0.066249,0.093017,2.595144,0.999413,True,False
2,lognormal,"{'mu': 2.0611602493254964, 'sigma': 0.95605398...","{'mu': [None, None], 'sigma': [0, None]}",2,-39835.966836,79675.933672,0.011569,0.009702,0.011569,0.021271,0.090711,0.999535,True,False
3,stretched_exponential,"{'Lambda': 1.4848207920308298, 'beta': 0.41033...","{'Lambda': [0, None], 'beta': [0, None]}",2,-39830.297156,79664.594313,0.011141,0.009991,0.011141,0.021132,0.066927,0.999578,True,False
4,truncated_power_law,"{'alpha': 2.2292261587339857, 'Lambda': 0.0149...","{'alpha': (0.0, None), 'Lambda': [0, None]}",2,-39823.933385,79651.866770,0.009233,0.009233,0.008469,0.017701,0.044438,0.999379,True,False


In [ ]:
from src.complex_networks.plots import plot_degree_distribution
from src.complex_networks.metrics import get_degree_distribution


plot_degree_distribution(
        get_degree_distribution(G),
        RESULTS_DIR / "teste.png",
        title="amazon0302",
        log_scale=True,
    )

In [14]:
get_degree_distribution(G)

Counter({5: 92835,
         6: 51356,
         7: 28534,
         8: 17845,
         9: 11754,
         10: 7975,
         4: 7284,
         3: 6774,
         2: 6148,
         1: 6069,
         11: 5819,
         12: 4069,
         13: 3003,
         14: 2218,
         15: 1747,
         16: 1346,
         17: 1081,
         18: 807,
         19: 702,
         20: 553,
         21: 465,
         22: 403,
         23: 336,
         24: 300,
         25: 252,
         26: 241,
         28: 178,
         29: 176,
         27: 175,
         30: 140,
         32: 110,
         31: 105,
         33: 86,
         34: 84,
         36: 79,
         35: 77,
         37: 57,
         39: 52,
         38: 50,
         46: 44,
         41: 43,
         40: 43,
         44: 41,
         42: 40,
         43: 33,
         50: 30,
         48: 29,
         56: 24,
         55: 24,
         45: 24,
         52: 22,
         51: 21,
         49: 20,
         47: 17,
         58: 16,
         59: 14,
   

In [2]:
from src.complex_networks.generation import sample_graphs

configs = {
    "exponential": {"Lambda": 0.20},
    "lognormal": {"mu": 3.0, "sigma": 0.45},
    "stretched_exponential": {"Lambda": 0.02, "beta": 1.5},
    "truncated_power_law": {"alpha": 1.2, "Lambda": 0.01},
}

graphs = sample_graphs(
    n=10_000,
    kmax=200,
    seed=42,
    configurations=configs
)


In [3]:
from src.complex_networks.plots import plot_degree_distribution
from src.complex_networks.metrics import get_degree_distribution


for name, graph in graphs.items():
    plot_degree_distribution(
        get_degree_distribution(graph),
        RESULTS_DIR / f"{name}.png",
        title=None,
        log_scale=True,
    )